In [1]:
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import json as json_lib 
from pub import Weak, Random, Real

In [2]:
op_dir = '../data/J123/'
data_dir = '../init/J/'
seed = 123

In [3]:
cols = ['user_id', 'exercise', 'correct', 'time_done'] 

log_df = pd.read_csv(os.path.join(data_dir, 'log.csv'), usecols=cols)
print(len(log_df), "\n")

log_df = log_df.sort_values(by='time_done')
log_df = log_df.drop_duplicates(subset=['user_id', 'exercise'], keep='first')
print(len(log_df), "\n")

log_df.rename(columns={'exercise': 'problem_id', 'time_done': 'order_id'}, inplace=True)
log_df.dropna(inplace=True)
print(len(log_df), "\n")

log_df['correct'] = log_df['correct'].astype(int)
print(log_df.head(), "\n")

25925992 

2289789 

2289789 

          user_id          problem_id          order_id  correct
15657986   111031  multiplication_0.5  1350003835131230        1
16566284   101006    area_of_a_circle  1350023005827490        1
5363433    109333      solid_geometry  1350037552915380        1
5631737    109333            angles_1  1350042486940170        1
13068659   109333         perimeter_1  1350043294983580        1 



In [4]:
exer_cols = ['name', 'prerequisites']

exer_df = pd.read_csv(os.path.join(data_dir, 'exercise.csv'), usecols=exer_cols)
print(len(exer_df), "\n")

exer_df.dropna(inplace=True)
print(len(exer_df), "\n")

exer_df.rename(columns={'name': 'problem_id', 'prerequisites': 'skill_ids'}, inplace=True)
print(exer_df.head(), "\n")

837 

742 

                        problem_id                                 skill_ids
0             parabola_intuition_1                recognizing_conic_sections
2               inscribed_angles_3                        inscribed_angles_2
3  solving_quadratics_by_factoring                   factoring_polynomials_1
4             graphing_parabolas_1                    graphing_parabolas_0.5
5            inverses_of_functions  domain_of_a_function,range_of_a_function 



In [5]:
full_df = pd.merge(log_df, exer_df, on='problem_id', how='inner')
full_df.sort_values(by='order_id', ascending=True, inplace=True)

print(len(full_df), "\n")
print(full_df.head(), "\n")

1976025 

   user_id          problem_id          order_id  correct  \
0   111031  multiplication_0.5  1350003835131230        1   
1   101006    area_of_a_circle  1350023005827490        1   
2   109333      solid_geometry  1350037552915380        1   
3   109333            angles_1  1350042486940170        1   
4   109333         perimeter_1  1350043294983580        1   

                                           skill_ids  
0                                         addition_2  
1  area_of_squares_and_rectangles,radius_diameter...  
2  areas_of_kites,areas_of_rhombi,areas_of_trapez...  
3                                     triangle_types  
4    order_of_operations,understand_range_of_graphie   



In [6]:
df = full_df.groupby('user_id').filter(lambda g: len(g) >= 15).copy()
print(len(df), "\n")

1360294 



In [7]:
sample_size = 10000
unique_users = df['user_id'].unique()

if len(unique_users) > sample_size:
    print(f"Sampling {sample_size} students from {len(unique_users)}...")
    np.random.seed(seed) 
    sampled_user_ids = np.random.choice(unique_users, size=sample_size, replace=False)
    df = df[df['user_id'].isin(sampled_user_ids)].copy()

print(f"Records after sampling: {len(df)}")
print(f"Actual students after sampling: {df['user_id'].nunique()}\n")


Sampling 10000 students from 32818...
Records after sampling: 409618
Actual students after sampling: 10000



In [8]:
def k2list(x):
    return [s.strip() for s in str(x).split(',')]

df['skill_ids'] = df['skill_ids'].apply(k2list)
print(len(df), "\n")
print(df.head(), "\n")

409618 

     user_id                       problem_id          order_id  correct  \
81    119053             representing_numbers  1350384641769170        1   
235   119053                    subtraction_1  1351084984074760        1   
236   119053                       addition_2  1351092811981810        1   
237   119053                    number_line_2  1351092973282940        1   
238   119053  number_properties_terminology_1  1351100723298060        0   

                                             skill_ids  
81   [comparison_between_numbers_within_ten, number...  
235                                       [addition_1]  
236  [number_line, addition_1, count_number_to_100,...  
237                             [subtracting_decimals]  
238        [multiplying_and_dividing_negative_numbers]   



In [9]:
s_all = df['user_id'].unique()
e_all = df['problem_id'].unique()
k_all = set([k for ks in df['skill_ids'] for k in ks])
    
s2idx = {uid: i for i, uid in enumerate(s_all)}
e2idx = {pid: i for i, pid in enumerate(e_all)}
k2idx = {sid: i for i, sid in enumerate(k_all)}

n, m, k = len(s2idx), len(e2idx), len(k2idx)

# 2 ID Mapping

In [10]:
df['user_id'] = df['user_id'].map(s2idx)
df['problem_id'] = df['problem_id'].map(e2idx)
df['skill_ids'] = df['skill_ids'].apply(lambda ks: [k2idx[s] for s in ks if s in k2idx])

print(len(df), "\n")
print(df.head(), "\n")

409618 

     user_id  problem_id          order_id  correct  \
81         0           0  1350384641769170        1   
235        0           1  1351084984074760        1   
236        0           2  1351092811981810        1   
237        0           3  1351092973282940        1   
238        0           4  1351100723298060        0   

                       skill_ids  
81                    [267, 157]  
235                         [14]  
236  [475, 14, 203, 35, 120, 48]  
237                         [77]  
238                        [367]   



In [11]:
p2k = df.groupby('problem_id')['skill_ids'].first().to_dict()
p2k_to_save = pd.DataFrame(p2k.items(), columns=['problem_id', 'skill_ids'])
p2k_to_save.to_csv(os.path.join(op_dir, 'p2k.csv'), index=False)

In [12]:
n_records = len(df)
avg_k_per_e = np.mean([len(ks) for ks in p2k.values()])
avg_r_per_s = n_records / n
n_correct = df[df['correct'] == 1].shape[0]
n_incorrect = df[df['correct'] == 0].shape[0]

print("\n## Dataset Statistics (Junyi)")
print(f"- records: {n_records}")
print(f"- students: {n}")
print(f"- questions: {m}")
print(f"- knowledge concepts: {k}")
print(f"- concepts per exercise: {avg_k_per_e:.4f}")
print(f"- records per student: {avg_r_per_s:.2f}")
print(f"- correct records / incorrect records: {n_correct} / {n_incorrect}")


## Dataset Statistics (Junyi)
- records: 409618
- students: 10000
- questions: 675
- knowledge concepts: 510
- concepts per exercise: 1.3556
- records per student: 40.96
- correct records / incorrect records: 277390 / 132228


In [13]:
meta = {
    'n': int(n), 
    'm': int(m), 
    'k': int(k),
    's2idx': {str(k): int(v) for k, v in s2idx.items()},
    'e2idx': {str(k): int(v) for k, v in e2idx.items()},
    'k2idx': {str(k): int(v) for k, v in k2idx.items()}
}
with open(os.path.join(op_dir, "meta.json"), 'w') as f:
    json.dump(meta, f)

In [14]:
kc_counts_series = pd.Series(index=df.index, dtype=str)

for user_id, group in tqdm(df.groupby('user_id'), "Generating kc_counts"):
    kc_counter = np.zeros(k, dtype=np.int16)
    for index, row in group.iterrows():
        kc_counts_series.at[index] = json_lib.dumps(kc_counter.tolist())
        k_practiced = row['skill_ids']
        if k_practiced:
            kc_counter[k_practiced] += 1
    
df['kc_counts'] = kc_counts_series
df.drop(columns=['skill_ids'], inplace=True)
print(df.head(), "\n")

Generating kc_counts: 100%|██████████| 10000/10000 [00:27<00:00, 360.90it/s]

     user_id  problem_id          order_id  correct  \
81         0           0  1350384641769170        1   
235        0           1  1351084984074760        1   
236        0           2  1351092811981810        1   
237        0           3  1351092973282940        1   
238        0           4  1351100723298060        0   

                                             kc_counts  
81   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
235  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
236  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...  
237  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, ...  
238  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, ...   



## (1) Weak Coverage Split

In [15]:
weak_splitter = Weak(op_dir, seed)
weak_splitter.split_and_save(df, p2k)


========== 1 Weak-Split (Corrected) ==========


Weak Split: 100%|██████████| 10000/10000 [00:06<00:00, 1653.32it/s]


Weak response proportion: 0.828
Weak Split Done. tv: 331448, test: 78170


## (2) Random 5-Fold Split

In [16]:
random_splitter = Random(op_dir, n_folds=5, seed=seed)
random_splitter.split_and_save(df, p2k)


========== 2 Random 5-Fold Split ==========


Random Split: 100%|██████████| 10000/10000 [00:07<00:00, 1320.75it/s]


Random 5-Fold Split Done. Avg Weak response proportion: 0.505


## (3) Real Scenario (Time-Series) Split

In [17]:
real_splitter = Real(op_dir=op_dir)
real_splitter.split_and_save(df, p2k)


========== 3 Real-Scenario (Time-Series) Split ==========


Time-Series Split: 100%|██████████| 10000/10000 [00:00<00:00, 13570.89it/s]


Weak response proportion: 0.770
Real Split Done. Train: 241905, Val: 78170, Test: 89543
